In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries loaded successfully!")


Matplotlib is building the font cache; this may take a moment.


All libraries loaded successfully!


In [3]:
df = pd.read_csv("data/PS_20174392719_1491204439457_log.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
fraud_counts = df['isFraud'].value_counts()
fraud_percentage = df['isFraud'].value_counts(normalize=True) * 100

print("Fraud vs Non-Fraud counts:")
print(fraud_counts)
print("\nFraud vs Non-Fraud percentage:")
print(fraud_percentage)

Fraud vs Non-Fraud counts:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud vs Non-Fraud percentage:
isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64


In [5]:
fraud_by_type = df[df['isFraud'] == 1]['type'].value_counts()
print("Fraud transactions by type:")
print(fraud_by_type)

print("\nAll transaction types in dataset:")
print(df['type'].value_counts())

Fraud transactions by type:
type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

All transaction types in dataset:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [6]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

Missing values per column:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Data types:
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


In [7]:
# 1. Drop isFlaggedFraud - this is PaySim's own rule-based flag, not something 
#    we should use as an input feature (it would be "cheating" / data leakage 
#    since it's essentially another fraud label, not a real predictive signal)
df_model = df.drop(columns=['isFlaggedFraud'])

# 2. Engineer a feature well-known to be predictive in this dataset:
#    the discrepancy between what SHOULD happen to the sender's balance
#    and what actually happened. Fraudulent transfers often show inconsistent
#    balances (e.g., balance doesn't decrease by the transaction amount).
df_model['errorBalanceOrig'] = df_model['oldbalanceOrg'] - df_model['amount'] - df_model['newbalanceOrig']
df_model['errorBalanceDest'] = df_model['oldbalanceDest'] + df_model['amount'] - df_model['newbalanceDest']

# 3. Extract whether the destination account is a merchant (starts with 'M')
#    or a regular customer (starts with 'C') - fraud typically targets customer accounts
df_model['isDestMerchant'] = df_model['nameDest'].str.startswith('M').astype(int)

# 4. One-hot encode the 'type' column - converts text categories into separate
#    binary (0/1) columns, since ML models need numbers, not text
df_model = pd.get_dummies(df_model, columns=['type'], prefix='type')

# 5. Drop the raw account ID columns - these are unique identifiers (like 
#    C1231006815), not useful patterns for the model to learn from directly
df_model = df_model.drop(columns=['nameOrig', 'nameDest'])

print("Shape after feature engineering:", df_model.shape)
df_model.head()

Shape after feature engineering: (6362620, 15)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,errorBalanceOrig,errorBalanceDest,isDestMerchant,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0.0,9839.64,1,False,False,False,True,False
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0.0,1864.28,1,False,False,False,True,False
2,1,181.00,181.0,0.00,0.0,0.0,1,0.0,181.00,0,False,False,False,False,True
3,1,181.00,181.0,0.00,21182.0,0.0,1,0.0,21363.00,0,False,True,False,False,False
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0.0,11668.14,1,False,False,False,True,False


In [8]:
from sklearn.model_selection import train_test_split

# Separate features (X) from the target we want to predict (y)
X = df_model.drop(columns=['isFraud'])
y = df_model['isFraud']

# Split into 80% training, 20% testing
# stratify=y ensures both sets keep the same fraud/non-fraud ratio (0.13%)
# - crucial given our severe class imbalance, otherwise a random split
# could easily produce a test set with very few or zero fraud examples
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nFraud % in training set:", y_train.mean() * 100)
print("Fraud % in test set:", y_test.mean() * 100)

Training set shape: (5090096, 14)
Test set shape: (1272524, 14)

Fraud % in training set: 0.12907418642005966
Fraud % in test set: 0.12911347840983745


In [9]:
# scale_pos_weight tells XGBoost how much more to "pay attention" to the 
# rare fraud class. Without this, the model would be biased toward always 
# predicting "not fraud" since that's 99.87% of the data.
# Standard formula: (number of negative cases) / (number of positive cases)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',   # area under precision-recall curve - better than
                             # accuracy for imbalanced problems like this
    random_state=42
)

model.fit(X_train, y_train)

print("Model training complete!")

scale_pos_weight: 773.7482496194825
Model training complete!


In [10]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, average_precision_score

# Get predictions on the test set (data the model has never seen)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # probability of fraud

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\n=== Key Scores ===")
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("Average Precision (PR-AUC):", average_precision_score(y_test, y_pred_proba))

=== Classification Report ===
              precision    recall  f1-score   support

   Not Fraud       1.00      1.00      1.00   1270881
       Fraud       0.99      1.00      0.99      1643

    accuracy                           1.00   1272524
   macro avg       0.99      1.00      1.00   1272524
weighted avg       1.00      1.00      1.00   1272524


=== Confusion Matrix ===
[[1270859      22]
 [      4    1639]]

=== Key Scores ===
ROC-AUC: 0.9994658324739233
Average Precision (PR-AUC): 0.9984362000632907


In [11]:
import joblib
import os

os.makedirs("../model", exist_ok=True)
joblib.dump(model, "../model/xgboost_fraud_model.pkl")

# Also save the exact list and order of feature columns - critical because
# when we score new transactions later, we must feed features in the same order
feature_names = list(X_train.columns)
joblib.dump(feature_names, "../model/feature_names.pkl")

print("Model saved to ml-service/model/xgboost_fraud_model.pkl")
print("Feature names saved:", feature_names)

Model saved to ml-service/model/xgboost_fraud_model.pkl
Feature names saved: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig', 'errorBalanceDest', 'isDestMerchant', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']
